# 任務 3：資料清理與處理決策
目標：比較刪除與填補，保留原始資料並檢查處理影響。


## 開始之前
姓名：歐國澤 學號：C114118111
### 學生操作方式
1. 先執行下一個完整的資料準備儲存格。每份任務可獨立執行；重啟環境後請由上而下重新執行。
2. 在每個小節的程式中，將 `___` 替換成符合題意的程式碼，並把 `# 學生註解` 改成自己的繁體中文解說（至少說明指定程式的輸入／處理／輸出）。`___` 是刻意留下的空格，完成前執行會報錯。
3. 先記錄預測，執行後完成驗證與解釋；再完成延伸練習。遇到困難可以使用小節專屬 AI 提示詞，仍須檢查、修改、執行與理解。
4. 在文字儲存格中填寫答案，保留程式、執行結果、圖表和 AI 使用紀錄。依教師指示交付已完成的 `.ipynb`。首次執行任務 1 的存檔程式會覆寫同名 practice_student_data.csv，請先確認或改檔名。


In [ ]:
# 載入分析與繪圖套件；np、pd、plt 是慣用縮寫
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 固定種子及呼叫順序，使本課範例能重現
rng = np.random.default_rng(42)
n = 60  # 模擬學生人數
departments = ["資訊管理", "企業管理", "金融管理"]
# 字典每個鍵是欄位名稱，每個值提供 60 位學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)  # 把字典轉成資料表
noise = rng.normal(0, 5, n)  # 模擬其他未觀察的差異
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)  # 限制 0 到 100 分
df.loc[[5, 17], "sleep_hours"] = np.nan  # 成績算完後才加入缺失
df.loc[8, "screen_time_hours"] = 15.0  # 成績算完後才加入異常值
print("資料已準備：", df.shape)

## 3.1 保留原始資料
`copy()` 建立獨立副本；處理前保留原始資料，才能追查與比較。

### 學生練習 3.1：完成程式與註解

**我的預測**複製出來的 df_clean 跟原本 df 一樣，sleep_hours 都是少 2 筆。

**AI求助提示詞：**
> 我是 Python 初學者，正在完成資料分析任務 3.1。我的資料表叫 df，請教我如何補上建立獨立資料副本的指令。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 輸入原始資料表 df，用 .copy() 複製出獨立副本，存給 df_clean
df_clean = df.copy()
print("原始缺失數：", df["sleep_hours"].isna().sum())
print("副本缺失數：", df_clean["sleep_hours"].isna().sum())
# isna().sum() 用來確認複製前後 sleep_hours 的缺失筆數是不是一致


### 驗證、解釋與延伸 3.1


**延伸練習的預測與驗證**預測兩張表的 shape 都是 `(60, 7)`，實際印出來確實都是 `(60, 7)`。

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：比較 df 和 df_clean 的列數與欄數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 輸入兩張資料表，用 .shape 印出各自的列數跟欄數來比對
print("原始大小：", df.shape, "副本大小：", df_clean.shape)


## 3.2 排除與填補
針對特定欄位排除缺失，並不是刪除原始資料。中位數填補保留筆數，但估計值會改變分布，也不會還原原始睡眠值。
**預測：** 兩種方案各保留幾筆？

### 學生練習 3.2：完成程式與註解

**我的預測**直接排除缺失會剩 58 筆，用中位數填補則會保留全部 60 筆。

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 3.2。我的資料表叫 df，請教我如何補上使用中位數填補副本 sleep_hours 的運算。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 輸入副本 sleep_hours 欄位與中位數 sleep_median，用 fillna() 填補空值後存回該欄
sleep_complete = df.dropna(subset=["sleep_hours"])
sleep_median = df_clean["sleep_hours"].median()
df_clean["sleep_hours"] = df_clean["sleep_hours"].fillna(sleep_median)
print("原始筆數：", len(df), "排除後：", len(sleep_complete))
print("填補中位數：", sleep_median)
print("原始與填補後平均：", df["sleep_hours"].mean(), df_clean["sleep_hours"].mean())
print("原始缺失數：", df["sleep_hours"].isna().sum())
# dropna(subset=["sleep_hours"]) 把睡眠時間缺漏的那幾列剔除，存成新資料表


### 驗證、解釋與延伸 3.2

**延伸練習的預測與驗證**預測用平均數填補後，缺失數一樣是 0，且整欄平均值會跟原始平均值一樣。實際執行後確認缺失為 0，平均值也與原始一致。

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：另建副本，以平均數填補睡眠缺失；比較中位數方案的平均值與缺失數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 建新副本並用平均數填補 sleep_hours 缺失，再跟中位數方案比對平均與缺失數
df_mean = df.copy()
df_mean["sleep_hours"] = df_mean["sleep_hours"].fillna(df_mean["sleep_hours"].mean())
print("平均數方案平均：", df_mean["sleep_hours"].mean(), "缺失數：", df_mean["sleep_hours"].isna().sum())
print("中位數方案平均：", df_clean["sleep_hours"].mean(), "缺失數：", df_clean["sleep_hours"].isna().sum())


## 3.3 處理異常
本案例無法從現有資料還原學生 9 在計分時的螢幕時間。可保留並註記，或另建篩選後資料作比較，避免憑空改值。

### 學生練習 3.3：完成程式與註解

**我的預測**篩選螢幕時間 <= 10 小時後會保留 59 筆，只排除掉學號 9 那位異常值。

**AI求助提示詞：**
> 我是 Python 初學者，正在完成資料分析任務 3.3。我的資料表叫 df，請教我如何補上利用 within_range 建立篩選後副本。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 輸入資料表 df 與布林條件 within_range，篩選出合理範圍資料並複製給 df_screen_filtered
within_range = df["screen_time_hours"] <= 10
df_screen_filtered = df[within_range].copy()
print("保留筆數：", len(df_screen_filtered))
print("被排除的學生：", df.loc[~within_range, "student_id"].tolist())
# ~within_range 把條件反轉，找出超過 10 小時的學生編號並轉成清單印出


### 驗證、解釋與延伸 3.3

**延伸練習的預測與驗證**預測 59 筆加上排除的 1 筆會等於原本的 60 筆，實際執行驗證結果為 `True`。

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：核對原始筆數是否等於保留加排除筆數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 計算保留筆數加排除筆數，跟原始筆數 len(df) 比對是否相等
excluded_count = (~within_range).sum()
print("是否相等：", len(df) == len(df_screen_filtered) + excluded_count)


## 任務交付與 AI 使用紀錄

- 使用的 AI 提示詞：用講義給的 3.1～3.3 提示詞查 `.copy()`、`fillna()` 跟布林條件篩選副本的寫法。
- AI 建議中我修改或拒絕的一處及原因：把 AI 原本寫得太長太制式的註解精簡掉，改用比較口語、直接講輸入／處理／輸出的短句。
- 遇到的錯誤或疑問、修正與驗證方式：一開始疑問篩選異常值時為什麼要多加 `.copy()`，查證後知道是為了避免修改新表時跳出 SettingWithCopyWarning 警告，加上後執行就正常了。
- 本任務一項有數據支持的發現及對應儲存格：3.2 節執行結果顯示，直接排除缺失會剩下 58 筆，用中位數填補則能保留全部 60 筆且缺失數降為 0。
- 一項資料或推論限制：填補進去的中位數或平均數都是估計值，雖然能保留筆數，但會改變原本資料的分布，並不是學生真實的睡眠時間。

**交付前檢核：** 已填完程式空格與註解；已填寫預測、驗證、解釋與延伸；已由上到下執行並確認圖表和數字與文字一致。
